# Bengaluru Climate Risk Scanner
## Notebook 06: Prepare the data for GeoServer and the web app

The last Colab step. It does three small jobs, then everything moves to your own computer (Phase 6 GeoServer, Phase 7 web app):

1. **An empty `properties` layer** in the GeoPackage. The web app will write uploaded properties into it through **WFS-T**, so it must exist with the right columns before GeoServer publishes it.
2. **A sample portfolio** of 500 made-up property loans (`sample_portfolio.csv`), placed on built-up land, to demo the upload without real bank data.
3. **One zip file** with the GeoPackage and every raster GeoServer needs, to download in one go.

Run it after Notebooks 01 to 05.

## 1. Connect Google Drive and load the configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, shutil
PROJECT_DIR = '/content/drive/MyDrive/bengaluru-climate-risk'
with open(f'{PROJECT_DIR}/config.json') as f:
    CONFIG = json.load(f)
RAW, OUT, GPKG = CONFIG['paths']['raw'], CONFIG['paths']['processed'], CONFIG['paths']['gpkg']
WORK = '/content/work'
os.makedirs(WORK, exist_ok=True)

## 2. Create the empty `properties` layer
Columns match the MySQL `properties` table (Phase 7), so the two can be joined on `portfolio_id` + `property_id`:

| Column | Type | Filled by |
|---|---|---|
| `geom` | Point, EPSG:4326 | Upload (latitude, longitude) |
| `portfolio_id` | integer | Web app |
| `property_id` | text | Upload |
| `loan_value` | decimal (₹) | Upload |
| `flood_class`, `heat_class` | 1 Low, 2 Medium, 3 High | Web app, from WFS queries on `flood_zones` and `heat_zones` |
| `risk_score` | decimal | Web app: 0.6 × flood + 0.4 × heat |
| `risk_level` | Low / Medium / High | Web app (High whenever flood is High) |

If a `properties` layer already exists it is replaced (empty again), which is handy for resetting the demo.

In [ ]:
import geopandas as gpd
import pandas as pd

LOCAL_GPKG = f'{WORK}/bengaluru.gpkg'
shutil.copy(GPKG, LOCAL_GPKG)

empty = gpd.GeoDataFrame({
    'portfolio_id': pd.Series(dtype='int32'),
    'property_id':  pd.Series(dtype='str'),
    'loan_value':   pd.Series(dtype='float64'),
    'flood_class':  pd.Series(dtype='int16'),
    'heat_class':   pd.Series(dtype='int16'),
    'risk_score':   pd.Series(dtype='float64'),
    'risk_level':   pd.Series(dtype='str'),
}, geometry=gpd.GeoSeries([], crs='EPSG:4326'))
empty.to_file(LOCAL_GPKG, layer='properties', driver='GPKG', geometry_type='Point')

shutil.copy(LOCAL_GPKG, GPKG)
print('Layers in the GeoPackage:')
print(gpd.list_layers(GPKG).to_string(index=False))

## 3. Generate a sample portfolio of 500 properties
Properties are placed at random on **built-up land inside the city** (built-up share of 50% or more), so they sit where real buildings are. Each gets a type and a made-up loan value typical for that type. A fixed random seed makes the file the same every time.

The CSV has exactly the columns the upload page will expect: `property_id, latitude, longitude, loan_value`, plus `property_type` for the dashboard.

In [ ]:
import numpy as np
import rasterio
from rasterio import features

N_PROPERTIES = 500
rng = np.random.default_rng(2022)

with rasterio.open(f'{RAW}/builtup_30m.tif') as src:
    builtup = src.read(1); transform, crs = src.transform, src.crs
with rasterio.open(f'{OUT}/flood_class.tif') as src:
    land = np.isin(src.read(1), [1, 2, 3])          # city land, not lakes

candidates = np.argwhere(land & (builtup >= 0.5))
pick = candidates[rng.choice(len(candidates), N_PROPERTIES, replace=False)]
# random position inside each chosen 30 m cell
xs, ys = rasterio.transform.xy(transform, pick[:, 0], pick[:, 1], offset='ul')
res = transform.a
xs = np.asarray(xs) + rng.uniform(0, res, N_PROPERTIES)
ys = np.asarray(ys) - rng.uniform(0, res, N_PROPERTIES)
pts = gpd.GeoSeries(gpd.points_from_xy(xs, ys), crs=crs).to_crs('EPSG:4326')

TYPES = {  # type: (share of portfolio, median loan in rupees, spread)
    'Home':        (0.45, 6_000_000,     0.5),
    'Apartment':   (0.30, 4_500_000,     0.4),
    'Retail':      (0.10, 20_000_000,    0.6),
    'Office':      (0.10, 50_000_000,    0.7),
    'IT park':     (0.05, 500_000_000,   0.6),
}
names = list(TYPES)
ptype = rng.choice(names, N_PROPERTIES, p=[TYPES[t][0] for t in names])
loan = np.array([rng.lognormal(np.log(TYPES[t][1]), TYPES[t][2]) for t in ptype])
loan = (np.round(loan / 10_000) * 10_000).astype('int64')      # round to ₹10,000

portfolio = pd.DataFrame({
    'property_id': [f'P-{i:04d}' for i in range(1, N_PROPERTIES + 1)],
    'latitude': pts.y.round(6).values,
    'longitude': pts.x.round(6).values,
    'loan_value': loan,
    'property_type': ptype,
})
csv_path = f'{OUT}/sample_portfolio.csv'
portfolio.to_csv(csv_path, index=False)
print('Saved', csv_path)
print(portfolio.head().to_string(index=False))
print(f'\nTotal loan value: ₹{portfolio.loan_value.sum() / 1e7:,.1f} crore')
print(portfolio.groupby('property_type')['loan_value'].agg(['count', 'median']).astype('int64').to_string())

## 4. Quick preview: where the sample properties fall on the flood map
A sanity check before the web app does this properly: the predicted flood class at each sample property.

In [ ]:
with rasterio.open(f'{OUT}/flood_class.tif') as src:
    fc = src.read(1)
rows, cols = rasterio.transform.rowcol(transform, xs, ys)
cls = fc[np.asarray(rows), np.asarray(cols)]
preview = portfolio.assign(flood=pd.Series(cls).map({1: 'Low', 2: 'Medium', 3: 'High'}))
summary = preview.groupby('flood').agg(properties=('property_id', 'size'), loan_crore=('loan_value', lambda v: v.sum() / 1e7))
summary['loan_share_%'] = 100 * summary['loan_crore'] / summary['loan_crore'].sum()
print(summary.round(1).to_string())

## 5. Bundle everything GeoServer needs
Creates `geoserver_bundle.zip` in the project folder. Download it from Google Drive (right-click → Download) and unzip it to `C:\climaterisk\data` on your computer, as the GeoServer guide (`docs/geoserver_setup.md` in the GitHub repo) describes.

In [ ]:
import zipfile

FILES = ['bengaluru.gpkg', 'flood_risk.tif', 'flood_class.tif', 'flood_factor_scores.tif', 'flood_factor_values.tif',
         'heat_risk.tif', 'heat_class.tif', 'heat_values.tif', 'flood_2022_observed.tif', 'validation_result.tif',
         'sample_portfolio.csv']
bundle = f'{WORK}/geoserver_bundle.zip'
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in FILES:
        path = f'{OUT}/{f}'
        if os.path.exists(path):
            z.write(path, arcname=f)
            print(f'  added   {f:<28} {os.path.getsize(path) / 1e6:6.1f} MB')
        else:
            print(f'  MISSING {f}  (run the notebook that makes it)')
shutil.copy(bundle, f'{PROJECT_DIR}/geoserver_bundle.zip')
print(f'\nSaved {PROJECT_DIR}/geoserver_bundle.zip ({os.path.getsize(bundle) / 1e6:.1f} MB)')
print('\nColab work complete. Next: install GeoServer on your computer (Phase 6).')